# Image Compression Using Singular Value Decomposition (SVD)

This project demonstrates image compression using Singular Value Decomposition (SVD). Both grayscale and color images are compressed by retaining only the largest singular values. The reconstructed images are compared to evaluate the relationship between compression level and image quality.

## 1. Required Libraries

Import the libraries required for numerical computation, image processing, and visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

## 2. Helper Functions

Three helper functions are defined:

- `show()`: Displays an image and hides the plot axes.
- `openImage()`: Loads an image and converts it into a NumPy array.
- `compress()`: Applies SVD to a matrix and reconstructs it using only the first k singular values.

In [ ]:
def show(image, cmap=None):
    plt.imshow(image, cmap=cmap)
    plt.axis('off')
    plt.show()


def openImage(imagePath):
    imOrig = Image.open(imagePath)
    im = np.array(imOrig)
    return im


def compress(Matrix, k):
    u, s, v = np.linalg.svd(Matrix, full_matrices=False)

    leftSide = np.matmul(u[:, :k], np.diag(s[:k]))
    CompressedInner = np.matmul(leftSide, v[:k, :])

    Compressed = np.clip(CompressedInner, 0, 255).astype('uint8')
    return Compressed

## 3. Grayscale Image Compression

A grayscale image is loaded and represented as a two-dimensional matrix. SVD is applied to decompose the image matrix into its singular values and corresponding singular vectors.

The image is reconstructed using different values of k (4, 8, and 12), retaining only the largest singular values.

The approximate storage size of the compressed image is calculated as:

$$k(1+m+n)$$

where m and n represent the image dimensions and k is the number of retained singular values.

In [ ]:
# Load Grayscale Image
imOrig = Image.open('/content/img_1.jpg').convert('L')
im = np.array(imOrig)
(m, n) = im.shape

# Display Original Image
print('Original Grayscale Image:')
show(im, cmap='gray')

# Calculate Original Storage Size
originalSize = m * n
print('Original size:', originalSize)
print('********')

# Compress Image Using Different Values of k
for k in (4, 8, 12):
    print('k =', k)

    compressImage = compress(im, k)
    newImage = Image.fromarray(compressImage)

    # Calculate Compressed Storage Size
    compressedSize = k * (1 + m + n)

    print('Original size:', originalSize)
    print('Compressed size:', compressedSize)

    # Display Compressed Image
    show(newImage, cmap='gray')
    print('********')

## 4. Color Image Compression

A color image is represented by three separate channels: Red, Green, and Blue (RGB). Each channel is treated as an independent two-dimensional matrix.

SVD is applied independently to each channel. The compressed channels are then merged to reconstruct the color image.

The compression process is repeated for k = 5, 10, 20, 100, and 200.

The approximate storage size for the three compressed channels is calculated as:

$$3k(1+m+n)$$

In [ ]:
# Load Color Image
colorImage = Image.open('/content/img_2.jpg').convert('RGB')
colorArray = np.array(colorImage)
(m, n, channels) = colorArray.shape

# Display Original Color Image
print('Original Color Image:')
show(colorArray)

# Calculate Original Storage Size
originalSize = m * n * 3
print('Original size:', originalSize)
print('********')

# Compress Image Using Different Values of k
for k in (5, 10, 20, 100, 200):
    print('k =', k)

    # Separate RGB Channels
    ared = colorArray[:, :, 0]
    agreen = colorArray[:, :, 1]
    ablue = colorArray[:, :, 2]

    # Compress Each Channel Separately
    redCompressed = compress(ared, k)
    greenCompressed = compress(agreen, k)
    blueCompressed = compress(ablue, k)

    # Merge Compressed RGB Channels
    compressedColor = np.stack(
        (redCompressed, greenCompressed, blueCompressed), axis=2
    )

    newColorImage = Image.fromarray(compressedColor, mode='RGB')

    # Calculate Storage Sizes
    compressedSize = 3 * k * (1 + m + n)

    print('Original size:', originalSize)
    print('Compressed size:', compressedSize)

    # Display Compressed Color Image
    show(newColorImage)
    print('********')

## 5. Results and Comparison

The reconstructed grayscale and color images are displayed for different values of k.

- Smaller values of k require less storage but generally result in lower image quality and loss of fine details.
- Larger values of k preserve more image information and produce better visual quality.
- The approximate storage requirements of the compressed representations can be compared with those of the original images.

This demonstrates how SVD provides a trade-off between image compression and reconstruction quality.